In [5]:
import os

file_path = r"C:\Users\N-14\documents.py"

print("File exists:", os.path.exists(file_path))

with open(file_path, "r", encoding="utf-8") as f:
    print(f.read()[:5000])

File exists: True
"""Dataset: 10 long-text documents for the TF-IDF class activity."""

TITLES = [
    "AI in Healthcare",
    "Climate Change",
    "Basketball Strategy",
    "Italian Pasta",
    "Space Exploration",
    "Cybersecurity",
    "Music Theory",
    "Gardening Vegetables",
    "Economics Inflation",
    "Public Health",
]

DOCUMENTS = [
    """Artificial intelligence is transforming healthcare by helping doctors analyze medical images, predict disease risk, and personalize treatments. Machine learning models can scan thousands of patient records to find patterns that humans might miss. Hospitals use AI to reduce waiting times and improve diagnosis accuracy. However, AI systems require large, clean datasets and careful validation before clinical use. Privacy, bias, and accountability are major concerns. If used responsibly, AI can support clinicians rather than replace them, making healthcare faster, safer, and more equitable for diverse populations.""",

    """Climate cha

In [6]:

import os
import sys
import importlib.util

# Load the existing documents.py file
FILE_PATH = r"C:\Users\N-14\documents.py"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"Could not find: {FILE_PATH}")

spec = importlib.util.spec_from_file_location(
    "documents_module", FILE_PATH
)
documents_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(documents_module)

# Use the variables defined in documents.py
titles = documents_module.TITLES
documents = documents_module.DOCUMENTS

print(f"{len(documents)} documents loaded.")

for title, document in zip(titles, documents):
    print(f"  - {title}: {len(document.split())} words")

assert len(titles) == 10 and len(documents) == 10


10 documents loaded.
  - AI in Healthcare: 85 words
  - Climate Change: 83 words
  - Basketball Strategy: 85 words
  - Italian Pasta: 91 words
  - Space Exploration: 86 words
  - Cybersecurity: 88 words
  - Music Theory: 87 words
  - Gardening Vegetables: 89 words
  - Economics Inflation: 89 words
  - Public Health: 91 words


In [7]:

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(documents)

print("Matrix shape:", X.shape)
print("Vocabulary size:", len(vectorizer.get_feature_names_out()))


Matrix shape: (10, 537)
Vocabulary size: 537


In [8]:

import pandas as pd

df = pd.DataFrame(
    X.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=titles,
)

print("\nSample of the TF-IDF matrix (first 3 docs × 10 terms):")
print(df.iloc[:3, :10].round(3))

print("\nTop 5 TF-IDF terms per document:")
for title in titles:
    top5 = df.loc[title].sort_values(ascending=False).head(5)
    print(f"\n{title}:")
    for term, score in top5.items():
        print(f"  {term:<20s} {score:.3f}")



Sample of the TF-IDF matrix (first 3 docs × 10 terms):
                     ability  accelerate  access  accountability  accuracy  \
AI in Healthcare        0.00         0.0     0.0           0.125     0.125   
Climate Change          0.00         0.0     0.0           0.000     0.000   
Basketball Strategy     0.13         0.0     0.0           0.000     0.000   

                     accurate  actions  activity  adaptation  add  
AI in Healthcare        0.000    0.000       0.0       0.000  0.0  
Climate Change          0.125    0.125       0.0       0.125  0.0  
Basketball Strategy     0.000    0.000       0.0       0.000  0.0  

Top 5 TF-IDF terms per document:

AI in Healthcare:
  ai                   0.374
  healthcare           0.249
  use                  0.165
  waiting              0.125
  transforming         0.125

Climate Change:
  emissions            0.250
  change               0.250
  climate              0.212
  energy               0.212
  smart                0.125

In [14]:

from sklearn.metrics.pairwise import cosine_similarity

def rank_documents(query: str):
    """Print documents ranked by cosine similarity to the query."""
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, X).flatten()

    ranking = sorted(
        zip(titles, scores),
        key=lambda item: item[1],
        reverse=True
    )

    print(f"\n=== Query: {query!r} ===")
    for title, score in ranking:
        print(f"  {title:<25s} {score:.3f}")

    return ranking


In [15]:

queries = [
    "renewable energy and climate policy",
    "password security and phishing",
    "healthy sleep exercise and diet",
]

for query in queries:
    rank_documents(query)



=== Query: 'renewable energy and climate policy' ===
  Climate Change            0.323
  Economics Inflation       0.095
  Gardening Vegetables      0.048
  AI in Healthcare          0.000
  Basketball Strategy       0.000
  Italian Pasta             0.000
  Space Exploration         0.000
  Cybersecurity             0.000
  Music Theory              0.000
  Public Health             0.000

=== Query: 'password security and phishing' ===
  Cybersecurity             0.176
  AI in Healthcare          0.000
  Climate Change            0.000
  Basketball Strategy       0.000
  Italian Pasta             0.000
  Space Exploration         0.000
  Music Theory              0.000
  Gardening Vegetables      0.000
  Economics Inflation       0.000
  Public Health             0.000

=== Query: 'healthy sleep exercise and diet' ===
  Public Health             0.285
  Gardening Vegetables      0.045
  AI in Healthcare          0.000
  Climate Change            0.000
  Basketball Strategy       0.0

In [16]:

expected_results = {
    "renewable energy and climate policy": "Climate Change",
    "password security and phishing": "Cybersecurity",
    "healthy sleep exercise and diet": "Public Health",
}

for query, expected_title in expected_results.items():
    ranking = rank_documents(query)
    actual_title = ranking[0][0]

    status = "PASS" if actual_title == expected_title else "CHECK"

    print(
        f"Expected: {expected_title} | "
        f"Actual: {actual_title} | {status}"
    )



=== Query: 'renewable energy and climate policy' ===
  Climate Change            0.323
  Economics Inflation       0.095
  Gardening Vegetables      0.048
  AI in Healthcare          0.000
  Basketball Strategy       0.000
  Italian Pasta             0.000
  Space Exploration         0.000
  Cybersecurity             0.000
  Music Theory              0.000
  Public Health             0.000
Expected: Climate Change | Actual: Climate Change | PASS

=== Query: 'password security and phishing' ===
  Cybersecurity             0.176
  AI in Healthcare          0.000
  Climate Change            0.000
  Basketball Strategy       0.000
  Italian Pasta             0.000
  Space Exploration         0.000
  Music Theory              0.000
  Gardening Vegetables      0.000
  Economics Inflation       0.000
  Public Health             0.000
Expected: Cybersecurity | Actual: Cybersecurity | PASS

=== Query: 'healthy sleep exercise and diet' ===
  Public Health             0.285
  Gardening Vegetable

In [17]:

vectorizer_v2 = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_df=0.8,
    min_df=2,
)

X_v2 = vectorizer_v2.fit_transform(documents)

print("Original matrix shape:", X.shape)
print("Updated matrix shape:", X_v2.shape)

print(
    "Original vocabulary size:",
    len(vectorizer.get_feature_names_out())
)

print(
    "Updated vocabulary size:",
    len(vectorizer_v2.get_feature_names_out())
)


Original matrix shape: (10, 537)
Updated matrix shape: (10, 69)
Original vocabulary size: 537
Updated vocabulary size: 69


In [18]:
%pip install pytest

Note: you may need to restart the kernel to use updated packages.


In [19]:
%pip install scikit-learn pandas pytest

Note: you may need to restart the kernel to use updated packages.


In [21]:
import os

print("Current directory:", os.getcwd())
print("Files and folders:", os.listdir())

Current directory: C:\Users\N-14
Files and folders: ['.anaconda', '.conda', '.continuum', '.ipynb_checkpoints', '.ipython', '.jupyter', '.keras', '3D Objects', 'anaconda3', 'anaconda_projects', 'AppData', 'Application Data', 'Assignment_2_DSTT_complete_submittion.ipynb', 'attchedOne.ipynb', 'census+income+kdd.zip', 'census.tar.gz', 'censusincomeOne.ipynb', 'Contacts', 'Cookies', 'dataset_cleaned.csv', 'Desktop', 'Documents', 'documents.py', 'Downloads', 'Favorites', 'Links', 'Local Settings', 'Muhammad Osama - The Attached Dataset 2 - The Attached Dataset 2 (1).csv', 'Muhammad Osama - The Attached Dataset 2 - The Attached Dataset 2.csv', 'Music', 'My Documents', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{53b39e87-18c4-11ea-a811-000d3aa4692b}.TxR.0.regtrans-ms', 'NTUSER.DAT{53b39e87-18c4-11ea-a811-000d3aa4692b}.TxR.1.regtrans-ms', 'NTUSER.DAT{53b39e87-18c4-11ea-a811-000d3aa4692b}.TxR.2.regtrans-ms', 'NTUSER.DAT{53b39e87-18c4-11ea-a811-000d3aa4692b}.TxR.bl

In [22]:
import os

src_path = r"C:\Users\N-14\src"

print("Files inside src:")
print(os.listdir(src_path))

Files inside src:
['run_activity.py', 'test_tfidf_utils.py', 'tfidf_utils.py', '__init__.py']


In [24]:
import os

for filename in ["test_tfidf_utils.py", "run_activity.py", "tfidf_utils.py"]:
    path = os.path.join(r"C:\Users\N-14\src", filename)

    print(f"\n{'=' * 15} {filename} {'=' * 15}")

    with open(path, "r", encoding="utf-8") as f:
        print(f.read())


=============== test_tfidf_utils.py ===============
"""Basic unit tests for the TF-IDF activity utilities."""
import sys
import os

sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

from src.load_data import load_documents
from src.tfidf_utils import build_tfidf, rank_documents_for_query


def test_load_documents():
    titles, docs = load_documents()
    assert len(titles) == 10
    assert len(docs) == 10
    assert all(len(d.split()) > 30 for d in docs)  # "long text"


def test_tfidf_matrix_shape():
    _, docs = load_documents()
    vectorizer, matrix = build_tfidf(docs)
    assert matrix.shape[0] == 10
    assert matrix.shape[1] == len(vectorizer.get_feature_names_out())


def test_query_ranking_matches_expected_topic():
    titles, docs = load_documents()
    vectorizer, matrix = build_tfidf(docs)
    ranking = rank_documents_for_query(
        "renewable energy and climate policy", vectorizer, matrix, titles
    )
    assert ranking[0][0] == "C

In [25]:
import os
import sys
import importlib

PROJECT_ROOT = r"C:\Users\N-14"
SRC_PATH = os.path.join(PROJECT_ROOT, "src")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Check the required files
required_files = ["tfidf_utils.py", "load_data.py"]

for filename in required_files:
    path = os.path.join(SRC_PATH, filename)
    print(f"{filename}: {'FOUND' if os.path.isfile(path) else 'MISSING'}")

# Stop if the data module is missing
if not os.path.isfile(os.path.join(SRC_PATH, "load_data.py")):
    print("\nPlease run the cell that creates load_data.py first.")
else:
    import pytest
    from src.load_data import load_documents
    from src.tfidf_utils import build_tfidf, rank_documents_for_query

    def test_load_documents():
        titles, docs = load_documents()
        assert len(titles) == 10
        assert len(docs) == 10
        assert all(len(d.split()) > 30 for d in docs)

    def test_tfidf_matrix_shape():
        _, docs = load_documents()
        vectorizer, matrix = build_tfidf(docs)
        assert matrix.shape[0] == 10
        assert matrix.shape[1] == len(vectorizer.get_feature_names_out())

    def test_query_ranking_matches_expected_topic():
        titles, docs = load_documents()
        vectorizer, matrix = build_tfidf(docs)
        ranking = rank_documents_for_query(
            "renewable energy and climate policy",
            vectorizer,
            matrix,
            titles
        )
        assert ranking[0][0] == "Climate Change", (
            f"Got {ranking[0][0]} instead of Climate Change"
        )

    passed = 0
    tests = [
        test_load_documents,
        test_tfidf_matrix_shape,
        test_query_ranking_matches_expected_topic
    ]

    for test in tests:
        try:
            test()
            print(f"PASS: {test.__name__}")
            passed += 1
        except Exception as e:
            print(f"FAIL: {test.__name__}: {e}")

    print(f"\nFinal result: {passed}/{len(tests)} passed")

tfidf_utils.py: FOUND
load_data.py: FOUND


ModuleNotFoundError: No module named 'data'

In [26]:
from pathlib import Path

file_path = Path(r"C:\Users\N-14\documents.py")

print("File exists:", file_path.exists())

if file_path.exists():
    print(file_path.read_text(encoding="utf-8")[:2000])

File exists: True
"""Dataset: 10 long-text documents for the TF-IDF class activity."""

TITLES = [
    "AI in Healthcare",
    "Climate Change",
    "Basketball Strategy",
    "Italian Pasta",
    "Space Exploration",
    "Cybersecurity",
    "Music Theory",
    "Gardening Vegetables",
    "Economics Inflation",
    "Public Health",
]

DOCUMENTS = [
    """Artificial intelligence is transforming healthcare by helping doctors analyze medical images, predict disease risk, and personalize treatments. Machine learning models can scan thousands of patient records to find patterns that humans might miss. Hospitals use AI to reduce waiting times and improve diagnosis accuracy. However, AI systems require large, clean datasets and careful validation before clinical use. Privacy, bias, and accountability are major concerns. If used responsibly, AI can support clinicians rather than replace them, making healthcare faster, safer, and more equitable for diverse populations.""",

    """Climate cha

In [27]:
from pathlib import Path

src_path = Path(r"C:\Users\N-14\src")
load_data_file = src_path / "load_data.py"

code = '''
"""Utilities for loading the TF-IDF activity dataset."""

import sys
from pathlib import Path

PROJECT_PATH = str(Path(r"C:\\Users\\N-14"))

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

from documents import TITLES, DOCUMENTS

def load_documents():
    """Return document titles and their text."""
    return TITLES, DOCUMENTS
'''

load_data_file.write_text(code.strip() + "\n", encoding="utf-8")
print("Successfully updated:", load_data_file)

Successfully updated: C:\Users\N-14\src\load_data.py


In [28]:
import sys
import importlib

if r"C:\Users\N-14" not in sys.path:
    sys.path.insert(0, r"C:\Users\N-14")

import src.load_data as load_data_module
import src.tfidf_utils as tfidf_module

importlib.reload(load_data_module)
importlib.reload(tfidf_module)

from src.load_data import load_documents
from src.tfidf_utils import build_tfidf, rank_documents_for_query

def test_load_documents():
    titles, docs = load_documents()
    assert len(titles) == 10
    assert len(docs) == 10
    assert all(len(d.split()) > 30 for d in docs)

def test_tfidf_matrix_shape():
    _, docs = load_documents()
    vectorizer, matrix = build_tfidf(docs)
    assert matrix.shape[0] == 10
    assert matrix.shape[1] == len(vectorizer.get_feature_names_out())

def test_query_ranking_matches_expected_topic():
    titles, docs = load_documents()
    vectorizer, matrix = build_tfidf(docs)

    ranking = rank_documents_for_query(
        "renewable energy and climate policy",
        vectorizer,
        matrix,
        titles
    )

    assert ranking[0][0] == "Climate Change"

tests = [
    test_load_documents,
    test_tfidf_matrix_shape,
    test_query_ranking_matches_expected_topic
]

for test in tests:
    test()
    print(f"PASS: {test.__name__}")

print("\nAll 3 tests passed!")

PASS: test_load_documents
PASS: test_tfidf_matrix_shape
PASS: test_query_ranking_matches_expected_topic

All 3 tests passed!
